In [21]:

import pandas as pd 
import numpy as np


# data load
file = pd.read_csv("../datasets/process_pima.csv")
train = file.copy()
train.head(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6.0,148.0,72.0,35.0,169.0,33.60,0.627,50.0,1.0
1,1.0,85.0,66.0,29.0,58.6,26.60,0.351,31.0,0.0
2,8.0,183.0,64.0,25.8,164.6,23.30,0.672,32.0,1.0
3,1.0,89.0,66.0,23.0,94.0,28.10,0.167,21.0,0.0
4,0.0,137.0,40.0,35.0,168.0,43.10,2.288,33.0,1.0
5,5.0,116.0,74.0,20.6,102.8,25.60,0.201,30.0,0.0
6,3.0,78.0,50.0,32.0,88.0,31.00,0.248,26.0,1.0
7,10.0,115.0,77.6,34.4,132.6,35.30,0.134,29.0,0.0
8,2.0,197.0,70.0,45.0,543.0,30.50,0.158,53.0,1.0
9,8.0,125.0,96.0,26.4,165.8,34.92,0.232,54.0,1.0


In [22]:
# with the ackonwledgement of medicline, construct new features for the dataset to impove the performance of the model.
# pregnancies >=5 is considered as high risk group for diabetes, as multiple pregnancies can lead to gestational diabetes and increase the risk of developing type 2 diabetes later in life.
# train['Pregnancies_Risk'] = train['Pregnancies'].apply(lambda x: 1 if x >= 5 else 0)

# # age >=30 is considered as a risk factor for diabetes, as the risk increases with age.
# train['Age_Risk'] = train['Age'].apply(lambda x: 1 if x >= 30 else 0)

# # BMI >=30 is considered as obese, which is a major risk factor for diabetes.
# train['BMI_Risk'] = train['BMI'].apply(lambda x: 1 if x >= 30 else 0)

# # glucose >=140 mg/dL is considered as prediabetes, and >=200 mg/dL is considered as diabetes, so we can create a risk feature based on these thresholds.
# train['Glucose_Risk'] = train['Glucose'].apply(lambda x: 2 if x >= 200 else (1 if x >= 140 else 0))

# # skin thickness >=30mm is considered as a risk factor for diabetes, as it may indicate insulin resistance.
# train['SkinThickness_Risk'] = train['SkinThickness'].apply(lambda x: 1 if x >= 30 else 0)

# # diabetes pedigree function >=0.5 is considered as a risk factor for diabetes, as it indicates a strong family history of diabetes.
# train['Pedigree_Risk'] = train['DiabetesPedigreeFunction'].apply(lambda x: 1 if x >= 0.5 else 0)

# # combine the risk features into a single metabolic risk score, which can help capture the overall risk profile of the patient.
# train['Metabolic_Score'] = (train['Glucose_Risk'] + train['BMI_Risk'] + train['Age_Risk'] + train['Pedigree_Risk'] + train['SkinThickness_Risk'] + train['Pregnancies_Risk'])
# train['High_Metabolic_Risk'] = train['Metabolic_Score'].apply(lambda x: 1 if x >= 3 else 0)

# 医学中常见比值特征
# 避免除0，加入极小常量 eps
# 这些比值可反应胰岛素敏感性、肥胖分布、年龄风险等
eps = 1e-6
# 代谢敏感性与胰岛素抵抗
train['Glucose_Insulin_ratio'] = train['Glucose'] / (train['Insulin'] + eps)
train['Insulin_Glucose_ratio'] = train['Insulin'] / (train['Glucose'] + eps)

# 肥胖与年龄相关性
train['BMI_Age_ratio'] = train['BMI'] / (train['Age'] + eps)
train['Age_BMI_interaction'] = train['Age'] * train['BMI']

# 皮脂-体重比、血压-年龄比
train['Skin_BMI_ratio'] = train['SkinThickness'] / (train['BMI'] + eps)
train['BloodPressure_Age_ratio'] = train['BloodPressure'] / (train['Age'] + eps)

# 输出检查
display(train.head(10))
train.to_csv("../datasets/feature_engineered_pima.csv", index=False,header=True)



,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Glucose_Insulin_ratio,Insulin_Glucose_ratio,BMI_Age_ratio,Age_BMI_interaction,Skin_BMI_ratio,BloodPressure_Age_ratio
0,6.0,148.0,72.0,35.0,169.0,33.60,0.627,50.0,1.0,0.875740,1.141892,0.672000,1680.00,1.041667,1.440000
1,1.0,85.0,66.0,29.0,58.6,26.60,0.351,31.0,0.0,1.450512,0.689412,0.858064,824.60,1.090226,2.129032
2,8.0,183.0,64.0,25.8,164.6,23.30,0.672,32.0,1.0,1.111786,0.899454,0.728125,745.60,1.107296,2.000000
3,1.0,89.0,66.0,23.0,94.0,28.10,0.167,21.0,0.0,0.946809,1.056180,1.338095,590.10,0.818505,3.142857
4,0.0,137.0,40.0,35.0,168.0,43.10,2.288,33.0,1.0,0.815476,1.226277,1.306061,1422.30,0.812065,1.212121
5,5.0,116.0,74.0,20.6,102.8,25.60,0.201,30.0,0.0,1.128405,0.886207,0.853333,768.00,0.804687,2.466667
6,3.0,78.0,50.0,32.0,88.0,31.00,0.248,26.0,1.0,0.886364,1.128205,1.192308,806.00,1.032258,1.923077
7,10.0,115.0,77.6,34.4,132.6,35.30,0.134,29.0,0.0,0.867270,1.153043,1.217241,1023.70,0.974504,2.675862
8,2.0,197.0,70.0,45.0,543.0,30.50,0.158,53.0,1.0,0.362799,2.756345,0.575472,1616.50,1.475410,1.320755
9,8.0,125.0,96.0,26.4,165.8,34.92,0.232,54.0,1.0,0.753920,1.326400,0.646667,1885.68,0.756014,1.777778
